In [1]:
"""
==============================================================================
DM1 Validation Automation Engine

Author      : Sanmathi S
Technology  : Python, Pandas, TQDM
Version     : Production Release (24-Jul-2026)

Description
-----------
This automation tool validates DM1 fault data against a master reference
database and automatically determines qualification status based on
multiple validation rules.

Key Features
------------
• Automated DM1 fault validation
• Source Address verification
• SPN_FMI validation
• Description validation
• Pcode validation
• Intelligent mismatch detection
• Qualification status generation
• High-speed processing for large datasets
• Automated CSV output generation

Validation Logic
----------------
1. Exact Match Validation
2. Source Address Mismatch Detection
3. SPN_FMI + Description Validation
4. SPN_FMI + Pcode Validation
5. Description + Pcode Validation
6. Individual Field Matching
7. Unique Fault Identification

Output Columns
--------------
• VEE Status
• VEE Description
• Master Data Description
• VEE Remarks

Business Benefits
-----------------
• Eliminates manual DM1 verification
• Reduces validation effort
• Improves analysis accuracy
• Accelerates fault data review
• Standardizes validation process

Domain
------
Automotive Diagnostics | Telematics | DM1 Fault Analysis

==============================================================================
"""

# ---------------------------
# production code 
# ---------------------------

import time
import pandas as pd
from tqdm import tqdm


# ---------------------------
# HARDCODED FILE PATHS
# ---------------------------
MASTER_FILE = r"C:\Users\HTL_Sanmathi\Downloads\july23_DM1_files\Master file.csv"

INPUT_FILE = r"C:\Users\HTL_Sanmathi\Downloads\july23_DM1_files\Telematics_Bosch_180HP_DTC_LIST_V1_R1 1(DTC).csv"

OUTPUT_FILE = r"C:\Users\HTL_Sanmathi\Downloads\july23_DM1_files\july27_DM1_filesDTC_Output.csv"

# ---------------------------
# SAFE CSV READER
# ---------------------------
def read_file(file_path):
    try:
        return pd.read_csv(file_path, encoding="utf-8")
    except Exception:
        return pd.read_csv(file_path, encoding="latin1")


# ---------------------------
# MAIN VALIDATION ENGINE
# ---------------------------
def run_validation():

    print("\n⚡ DM1 VALIDATION ENGINE ⚡")
    print("=" * 40)
    print(f"📋 Master File : {MASTER_FILE}")
    print(f"📥 Input File  : {INPUT_FILE}")
    print(f"📤 Output File : {OUTPUT_FILE}")
    print("=" * 40)

    try:

        print("\n⏳ Loading files...")
        start_time = time.time()

        master_df = read_file(MASTER_FILE)
        input_df = read_file(INPUT_FILE)

        # ---------------------------
        # CLEAN COLUMN NAMES
        # ---------------------------
        master_df.columns = master_df.columns.str.strip()
        input_df.columns = input_df.columns.str.strip()

        # ---------------------------
        # REQUIRED COLUMNS CHECK
        # ---------------------------
        required_cols = [
            "SPN_FMI",
            "Source Address",
            "Description",
            "Pcode"
        ]

        for col in required_cols:
            if col not in master_df.columns:
                print(f"\n❌ Missing column in Master File : {col}")
                return

            if col not in input_df.columns:
                print(f"\n❌ Missing column in Input File : {col}")
                return

        # ---------------------------
        # CLEAN DATA
        # ---------------------------
        for col in required_cols:

            master_df[col] = (
                master_df[col]
                .astype(str)
                .str.strip()
                .str.upper()
            )

            input_df[col] = (
                input_df[col]
                .astype(str)
                .str.strip()
                .str.upper()
            )

        print("\n✅ Data Standardization Completed")

        # ---------------------------
        # MATCH FUNCTION
        # ---------------------------
        def get_match(row, cols):

            # Filter Master using current row Source Address
            df = master_df[
                master_df["Source Address"] == row["Source Address"]
            ]

            for col in cols:
                df = df[df[col] == row[col]]

            return df.iloc[0] if not df.empty else None

        # ---------------------------
        # VALIDATION
        # ---------------------------
        def validate_row(row):

            vee_desc = (
                f"SPN_FMI:{row['SPN_FMI']} | "
                f"Description:{row['Description']} | "
                f"Pcode:{row['Pcode']}"
            )

            master_desc = ""

            # -------------------------------------------------------------
            # 1. FIRST VALIDATION: Correct Source Address Mismatch Logic
            # -------------------------------------------------------------
            # Find rows where fault core fields match, but Source Address is different
            sa_mismatch_df = master_df[
                (master_df["SPN_FMI"] == row["SPN_FMI"]) &
                (master_df["Description"] == row["Description"]) &
                (master_df["Pcode"] == row["Pcode"]) &
                (master_df["Source Address"] != row["Source Address"])
            ]

            if not sa_mismatch_df.empty:
                match = sa_mismatch_df.iloc[0]
                master_desc = (
                    f"SPN_FMI:{match['SPN_FMI']} | "
                    f"Description:{match['Description']} | "
                    f"Pcode:{match['Pcode']}"
                )
                return pd.Series([
                    "Not Feasible",
                    vee_desc,
                    master_desc,
                    "Source Address mismatch"
                ])

            # -------------------------------------------------------------
            # 2. Exact Match (Same Source Address + All fields match)
            # -------------------------------------------------------------
            match = get_match(
                row,
                ["SPN_FMI", "Description", "Pcode"]
            )

            if match is not None:

                master_desc = (
                    f"SPN_FMI:{match['SPN_FMI']} | "
                    f"Description:{match['Description']} | "
                    f"Pcode:{match['Pcode']}"
                )

                return pd.Series([
                    "Qualified",
                    vee_desc,
                    master_desc,
                    "Exact match"
                ])

            # 3. SPN_FMI + Description
            match = get_match(
                row,
                ["SPN_FMI", "Description"]
            )

            if match is not None:

                master_desc = (
                    f"SPN_FMI:{match['SPN_FMI']} | "
                    f"Description:{match['Description']} | "
                    f"Pcode:{match['Pcode']}"
                )

                return pd.Series([
                    "Not Feasible",
                    vee_desc,
                    master_desc,
                    "SPN_FMI + Description match, Pcode mismatch"
                ])

            # 4. SPN_FMI + Pcode
            match = get_match(
                row,
                ["SPN_FMI", "Pcode"]
            )

            if match is not None:

                master_desc = (
                    f"SPN_FMI:{match['SPN_FMI']} | "
                    f"Description:{match['Description']} | "
                    f"Pcode:{match['Pcode']}"
                )

                return pd.Series([
                    "Not Feasible",
                    vee_desc,
                    master_desc,
                    "SPN_FMI + Pcode match, Description mismatch"
                ])

            # 5. Description + Pcode
            match = get_match(
                row,
                ["Description", "Pcode"]
            )

            if match is not None:

                master_desc = (
                    f"SPN_FMI:{match['SPN_FMI']} | "
                    f"Description:{match['Description']} | "
                    f"Pcode:{match['Pcode']}"
                )

                return pd.Series([
                    "Not Feasible",
                    vee_desc,
                    master_desc,
                    "Description + Pcode match, SPN_FMI mismatch"
                ])

            # 6. Only SPN_FMI
            match = get_match(
                row,
                ["SPN_FMI"]
            )

            if match is not None:

                master_desc = (
                    f"SPN_FMI:{match['SPN_FMI']} | "
                    f"Description:{match['Description']} | "
                    f"Pcode:{match['Pcode']}"
                )

                return pd.Series([
                    "Not Feasible",
                    vee_desc,
                    master_desc,
                    "Only SPN_FMI matches"
                ])

            # 7. Only Description
            match = get_match(
                row,
                ["Description"]
            )

            if match is not None:

                master_desc = (
                    f"SPN_FMI:{match['SPN_FMI']} | "
                    f"Description:{match['Description']} | "
                    f"Pcode:{match['Pcode']}"
                )

                return pd.Series([
                    "Not Feasible",
                    vee_desc,
                    master_desc,
                    "Only Description matches"
                ])

            # 8. Only Pcode
            match = get_match(
                row,
                ["Pcode"]
            )

            if match is not None:

                master_desc = (
                    f"SPN_FMI:{match['SPN_FMI']} | "
                    f"Description:{match['Description']} | "
                    f"Pcode:{match['Pcode']}"
                )

                return pd.Series([
                    "Not Feasible",
                    vee_desc,
                    master_desc,
                    "Only Pcode matches"
                ])

            return pd.Series([
                "Not Feasible",
                vee_desc,
                "",
                "Unique Pcode, SPN_FMI, Description"
            ])

        # ---------------------------
        # PROCESS
        # ---------------------------
        print("\n⚡ Processing validation...")

        tqdm.pandas(desc="Processing Rows")

        input_df[
            [
                "VEE Status",
                "VEE Description",
                "Master Data Description",
                "VEE Remarks"
            ]
        ] = input_df.progress_apply(validate_row, axis=1)

        # ---------------------------
        # SAVE FILE
        # ---------------------------
        print(f"\n💾 Saving Output to: {OUTPUT_FILE}")
        input_df.to_csv(OUTPUT_FILE, index=False)
        
        end_time = time.time()
        duration = round(end_time - start_time, 2)
        print(f"✅ Validation finished successfully in {duration} seconds!")

    except Exception as e:
        print(f"\n❌ An error occurred: {str(e)}")


if __name__ == "__main__":
    run_validation()



⚡ DM1 VALIDATION ENGINE ⚡
📋 Master File : C:\Users\HTL_Sanmathi\Downloads\july23_DM1_files\Master file.csv
📥 Input File  : C:\Users\HTL_Sanmathi\Downloads\july23_DM1_files\Telematics_Bosch_180HP_DTC_LIST_V1_R1 1(DTC).csv
📤 Output File : C:\Users\HTL_Sanmathi\Downloads\july23_DM1_files\july27_DM1_filesDTC_Output.csv

⏳ Loading files...

✅ Data Standardization Completed

⚡ Processing validation...


Processing Rows: 100%|██████████| 307/307 [00:02<00:00, 107.80it/s]


💾 Saving Output to: C:\Users\HTL_Sanmathi\Downloads\july23_DM1_files\july27_DM1_filesDTC_Output.csv
✅ Validation finished successfully in 2.95 seconds!
